<a href="https://colab.research.google.com/github/bsheese/387/blob/main/23_language/23_0_Tokens_and_Embeddings/23_0_1_Tokens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Language: Part 1
## Tokens

---

## What This Notebook Is About

Everything in the course so far has started from numbers: pixel values, measurements, features. Text isn't numbers, so before any network can read it, it has to be turned into numbers. This notebook covers the first step, **tokenization**: splitting text into pieces called **tokens** and giving each piece an ID number. 23_0_2 covers the second step, turning each ID into a vector of numbers that a network can learn from.

Unit 23 uses one dataset throughout: reviews of movies from the website IMDB, each labeled as positive or negative. By 23_2 you'll fine-tune a pretrained language model to tell the two apart, the way 19_0 and 22_1 fine-tuned an image model.

**What you will learn:**
1. Why text is split into **subword** tokens rather than characters or whole words
2. How a tokenizer turns a review into token IDs and back
3. How a batch of reviews of different lengths becomes one tensor, with **padding**, **truncation** and an **attention mask**
4. Why every model comes with its own tokenizer
5. Why token counts matter when you use large language models

This notebook downloads two tokenizers from Hugging Face, the main website for sharing pretrained language models. It doesn't need a GPU.

In [ ]:
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer

DATA_URL = 'https://raw.githubusercontent.com/bsheese/387/main/data/'

def fetch(name):
    # Download a file from the course's data folder, unless it is already here.
    if not os.path.exists(name):
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name

reviews = pd.read_csv(fetch('imdb_reviews.csv'))
print(reviews['split'].value_counts().to_dict())
reviews.head()

---

## Section 1: The Reviews

The file has 5,000 reviews from the Large Movie Review Dataset, which researchers at Stanford released in 2011. Like the other course datasets, it's a fixed subset, so everyone works with the same reviews: 2,000 for training, 1,000 for validation and 2,000 for testing, each half positive (`label` 1) and half negative (`label` 0). Here are two:

In [ ]:
train = reviews[reviews['split'] == 'train']
for label in [1, 0]:
    text = train[train['label'] == label]['text'].iloc[0]
    print(f"label {label} ({'positive' if label else 'negative'}):\n{text[:600]}...\n")

In [ ]:
lengths = reviews['text'].str.len()
print(f'review length in characters: median {lengths.median():.0f}, shortest {lengths.min()}, longest {lengths.max():,}')

The reviews vary a lot in length, from one sentence to several pages. They also contain everything real text contains: names, misspellings, slang, punctuation and numbers.

---

## Section 2: Characters, Words, or Something in Between?

There are two obvious ways to split text into pieces, and both have problems.

**Characters.** There are only about a hundred different characters in English text, so the list of possible pieces (the **vocabulary**) is tiny, and any text at all can be written with it. But a review becomes a sequence of over a thousand pieces, each of which means very little on its own; the network has to learn from scratch that "m", "o", "v", "i", "e" together mean something.

**Whole words.** Each piece carries meaning, and sequences are much shorter. But the vocabulary is enormous. Splitting the 2,000 training reviews at spaces shows the problem:

In [ ]:
words = ' '.join(train['text']).lower().split()
counts = pd.Series(words).value_counts()
print(f'{len(words):,} words in the training reviews, {len(counts):,} different ones')
print(f'{(counts == 1).sum():,} of the different words appear only once')
print('some of them:', list(counts[counts == 1].index[:12]))

More than half of the different "words" appear only once, and many are names, misspellings or words glued to punctuation. A network can't learn much about a word it sees once, and any word that never appeared in training, like a new actor's name, would have no number at all. Real vocabularies of this kind run to hundreds of thousands of entries and still miss words.

**Subwords** are the compromise used by modern language models. Common words are kept whole, and rarer words are split into pieces that are common: prefixes, suffixes and syllables. The vocabulary stays a manageable size (tens of thousands of pieces), every word can still be written with it, and related words share pieces.

---

## Section 3: A Real Tokenizer

Here is the tokenizer for **DistilBERT**, the model you'll fine-tune in 23_2. Hugging Face's `AutoTokenizer` downloads the right tokenizer for any model by name.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
print(f'vocabulary size: {tokenizer.vocab_size:,}')

for word in ['movie', 'movies', 'overrated', 'unbelievably', 'Tarantino', "didn't", 'supercalifragilistic']:
    print(f'{word:22} -> {tokenizer.tokenize(word)}')

- Common words like "movie" are a single token.
- Less common words are split into pieces. A piece that starts with `##` continues the word before it, so `un`, `##bel`, `##ie`, `##va`, `##bly` is one word in five tokens.
- Names are split too, so the tokenizer never meets a word it can't handle.
- "Uncased" in the model's name means everything is lowercased first, so "Tarantino" and "tarantino" are the same.

The vocabulary was built by starting from single characters and repeatedly adding the combination of pieces that appeared most often in a large amount of text, until it reached about 30,000 entries. (This version is called **WordPiece**.) So which words stay whole depends on how common they were in the text the vocabulary was built from.

---

## Section 4: Tokens to IDs and Back

Each token in the vocabulary has an ID number. Calling the tokenizer on some text gives the IDs:

In [ ]:
sentence = 'This movie was unbelievably good!'
encoded = tokenizer(sentence)
print('input_ids:', encoded['input_ids'])
print('tokens:   ', tokenizer.convert_ids_to_tokens(encoded['input_ids']))
print('decoded:  ', tokenizer.decode(encoded['input_ids']))

Two tokens were added that aren't in the sentence. `[CLS]` (ID 101) at the start and `[SEP]` (ID 102) at the end are **special tokens** that DistilBERT was trained with. `[SEP]` marks the end of a piece of text. `[CLS]` is a placeholder at the start whose final vector is often used to represent the whole text; 23_2 uses it to classify reviews. Decoding the IDs gives back the text, lowercased.

---

## Section 5: A Batch of Reviews as One Tensor

Networks work on batches (21_0_1), and a batch is one tensor, which must be rectangular. But reviews have different lengths. Two settings fix that:

- **Padding** adds a special `[PAD]` token (ID 0) to the end of shorter texts until every text in the batch is as long as the longest.
- **Truncation** cuts texts longer than a maximum number of tokens.

The tokenizer also returns an **attention mask**: 1 for real tokens and 0 for padding, so that the model can ignore the padding (23_1 shows how).

In [ ]:
batch_texts = ['Great movie!', 'The plot made no sense and the acting was wooden.', 'Meh.']
batch = tokenizer(batch_texts, padding=True, truncation=True, max_length=16, return_tensors='pt')
print('input_ids shape:', tuple(batch['input_ids'].shape))
print(batch['input_ids'])
print(batch['attention_mask'])

`return_tensors='pt'` returns PyTorch tensors. The batch has shape (3, 13): 3 texts, each padded to the length of the longest one, 13 tokens. From here on, text is just a tensor of whole numbers, like the labels in unit 21.

### How long are the reviews in tokens?

Every model has a maximum number of tokens it can read at once; for DistilBERT it's 512. Longer texts must be truncated, and the time a model takes grows quickly with the length (23_1 explains why), so in practice people often truncate further.

In [ ]:
token_counts = np.array([len(tokenizer(text)['input_ids']) for text in reviews['text']])
print(f'tokens per review: median {np.median(token_counts):.0f}, mean {token_counts.mean():.0f}, longest {token_counts.max():,}')
for limit in [128, 256, 512]:
    print(f'reviews longer than {limit} tokens: {(token_counts > limit).mean():.0%}')

plt.figure(figsize=(7, 3.5))
plt.hist(np.minimum(token_counts, 1200), bins=60)
for limit in [256, 512]:
    plt.axvline(limit, color='gray', linestyle='--')
plt.xlabel('tokens per review (longer reviews shown at 1,200)')
plt.ylabel('number of reviews')
plt.show()

(The tokenizer prints a warning while counting, because some reviews are longer than DistilBERT's 512-token limit. That's expected here; we're only counting.)

The median review is a little over 200 tokens. Truncating at 256 tokens keeps all of most reviews but cuts the end off about 40% of them. Whether that matters depends on the task: a reviewer's opinion is usually clear early, but some reviews end with "...but it's still terrible." 23_2 uses 256 tokens and checks the effect.

---

## Section 6: Every Model Has Its Own Tokenizer

A model learns one vector for each ID in its own vocabulary, so the IDs only mean something to that model. Here is the same sentence through **GPT-2**'s tokenizer, the text-generating model in 23_3:

In [ ]:
gpt2_tokenizer = AutoTokenizer.from_pretrained('gpt2')
print(f'GPT-2 vocabulary size: {gpt2_tokenizer.vocab_size:,}')

sentence = 'This movie was unbelievably good! Tarantino at his best.'
for name, tok in [('DistilBERT', tokenizer), ('GPT-2', gpt2_tokenizer)]:
    ids = tok(sentence)['input_ids']
    print(f'\n{name}: {len(ids)} tokens')
    print('  ', tok.convert_ids_to_tokens(ids))
    print('   ids:', ids[:8], '...')
print()
print('"movie" in DistilBERT:', tokenizer.encode('movie', add_special_tokens=False),
      '   " movie" in GPT-2:', gpt2_tokenizer.encode(' movie'))

GPT-2's tokenizer is different in several ways. Its vocabulary has about 50,000 entries, it keeps capital letters, it adds no special tokens, and it marks a space before a word as part of the token (shown as `Ġ`). The word "movie" has one ID in DistilBERT's vocabulary and a completely different one in GPT-2's. Feeding one model's IDs into another model would give nonsense, which is why Hugging Face loads a model and its tokenizer by the same name.

---

## Section 7: Why Token Counts Matter

Large language models are limited and priced in tokens. A model's **context window**, the most text it can consider at once, is a number of tokens, and commercial models charge per token read and written. So it's useful to know how words convert to tokens:

In [ ]:
sample = ' '.join(train['text'].iloc[:200])
n_words = len(sample.split())
n_tokens = len(gpt2_tokenizer(sample)['input_ids'])
print(f'{n_words:,} words -> {n_tokens:,} GPT-2 tokens  ({n_tokens / n_words:.2f} tokens per word)')

for language, text in [('English', 'The movie was wonderful and the actors were excellent.'),
                       ('Spanish', 'La película fue maravillosa y los actores fueron excelentes.')]:
    print(f"{language:8} {len(text.split())} words -> {len(gpt2_tokenizer(text)['input_ids'])} GPT-2 tokens")

For English movie reviews, GPT-2's tokenizer uses a bit more than one token per word on average, since punctuation and less common words take extra tokens. The same sentence in Spanish takes about twice as many tokens as in English. GPT-2's vocabulary was built mostly from English web pages, so common English words are single tokens while Spanish words are more often split into pieces. Languages less represented in a model's training text need more tokens for the same content. That means less fits in the context window and it costs more to process, which is a small, concrete example of the data issues from 19_0_2 showing up in language models.

---

## Summary

- Text has to become numbers before a network can use it. **Tokenization** splits it into **tokens** and gives each one an ID from a fixed **vocabulary**.
- Characters make sequences too long and each piece meaningless; whole words make the vocabulary huge and can't handle new words. **Subword** tokenizers keep common words whole and split rare words into common pieces, so any text can be written with about 30,000–50,000 tokens.
- DistilBERT's tokenizer (WordPiece, lowercased) adds the special tokens `[CLS]` and `[SEP]`; `##` marks a piece that continues a word.
- A batch of texts becomes one tensor of shape (batch, tokens) by **padding** short texts and **truncating** long ones. The **attention mask** marks which tokens are real.
- The median IMDB review is about 220 DistilBERT tokens; 14% are longer than DistilBERT's 512-token limit.
- Each model has its own tokenizer, and IDs from one are meaningless to another.
- Context windows and costs are counted in tokens. Text in languages that were less common in a tokenizer's training data takes more tokens.

**Next:** 23_0_2 turns token IDs into **embeddings**, vectors of numbers that a network learns, and looks at what those vectors capture.